# 低評価予測モデルの追加特徴量設計

## 目的

基準モデルで使用した

- total_price
- total_freight
- delivery_delay_days

に加えて、注文・顧客地域・購入時期・支払情報などの特徴量を検討し、
低評価予測性能が改善するかを確認する。

## 予測時点

本分析では、

「配送完了後、レビュー投稿前」

に低評価リスクを予測することを想定する。

そのため、配送完了時点までに取得可能な情報のみを特徴量として使用し、
レビュー内容やレビュー投稿後に得られる情報は使用しない。

## 追加特徴量候補

| 特徴量 | 分類 | 採用候補 |
|---|---|---|
| total_price | 注文金額 | 基準モデル |
| total_freight | 送料 | 基準モデル |
| delivery_delay_days | 配送 | 基準モデル |
| item_count | 注文構成 | 追加候補 |
| category_count | 注文構成 | 追加候補 |
| freight_ratio | 注文構成 | 追加候補 |
| customer_state | 地域 | 追加候補 |
| purchase_month | 時期 | 追加候補 |
| purchase_weekday | 時期 | 追加候補 |
| payment_type | 支払 | 追加候補 |
| payment_installments | 支払 | 追加候補 |

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src").is_dir() and (path / "notebooks").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.project_paths import DATA_DIR

df_review_order = pd.read_csv(DATA_DIR / "review_order_summary.csv")
df_delivery = pd.read_csv(DATA_DIR / "delivery_delay_review_summary.csv")
df_merge = pd.merge(
    df_review_order,
    df_delivery[["order_id", "delivery_delay_days"]],
    on="order_id", how="inner",
)
print(df_merge.shape)
print(df_merge.columns.tolist())
print(df_merge.head())

(95830, 5)
['order_id', 'review_score', 'total_price', 'total_freight', 'delivery_delay_days']
                           order_id  review_score  total_price  total_freight  \
0  0005a1a1728c9d785b8e2b08b904576c           1.0       145.95          11.65   
1  000aed2e25dbad2f9ddb70584c5a2ded           1.0       144.00           8.77   
2  00169e31ef4b29deaae414f9a5e95929           1.0        39.00          16.11   
3  0017afd5076e074a48f1f1a4c7bac9c5           1.0       809.10          44.29   
4  001d8f0e34a38c37f7dba2a37d4eba8b           1.0        37.98          15.56   

   delivery_delay_days  
0                    0  
1                   -4  
2                  -15  
3                    4  
4                    2  


In [3]:
df_order_state = pd.read_csv(DATA_DIR / "order_level_sales_by_state.csv")

print("低評価予測データ:", df_merge.shape)
print("売上注文データ:", df_order_state.shape)

print("売上注文データのorder_id重複数:", df_order_state["order_id"].duplicated().sum())

低評価予測データ: (95830, 5)
売上注文データ: (96478, 5)
売上注文データのorder_id重複数: 0


In [4]:
df_model_extended = df_merge.merge(
    df_order_state[
        ["order_id", "purchase_month", "customer_state", "order_gmv", "order_freight"]
    ],
    on="order_id",
    how="left",
)

print(df_model_extended.shape)
print(df_model_extended.head())


(95830, 9)
                           order_id  review_score  total_price  total_freight  \
0  0005a1a1728c9d785b8e2b08b904576c           1.0       145.95          11.65   
1  000aed2e25dbad2f9ddb70584c5a2ded           1.0       144.00           8.77   
2  00169e31ef4b29deaae414f9a5e95929           1.0        39.00          16.11   
3  0017afd5076e074a48f1f1a4c7bac9c5           1.0       809.10          44.29   
4  001d8f0e34a38c37f7dba2a37d4eba8b           1.0        37.98          15.56   

   delivery_delay_days purchase_month customer_state  order_gmv  order_freight  
0                    0        2018-03             SP     145.95          11.65  
1                   -4        2018-05             SP     144.00           8.77  
2                  -15        2018-01             RJ      39.00          16.11  
3                    4        2017-04             AL     809.10          44.29  
4                    2        2017-05             SP      37.98          15.56  


In [5]:
print(
    df_model_extended[
        ["purchase_month", "customer_state", "order_gmv", "order_freight"]
    ]
    .isna()
    .sum()
)


purchase_month    6
customer_state    6
order_gmv         6
order_freight     6
dtype: int64


In [6]:
print(
    "total_price と order_gmv の不一致:",
    (
        (df_model_extended["total_price"] - df_model_extended["order_gmv"]).abs() > 1e-6
    ).sum(),
)

print(
    "total_freight と order_freight の不一致:",
    (
        (df_model_extended["total_freight"] - df_model_extended["order_freight"]).abs()
        > 1e-6
    ).sum(),
)


total_price と order_gmv の不一致: 0
total_freight と order_freight の不一致: 0


In [7]:
df_model_extended = df_model_extended.drop(columns=["order_gmv", "order_freight"])

print(df_model_extended.columns.tolist())


['order_id', 'review_score', 'total_price', 'total_freight', 'delivery_delay_days', 'purchase_month', 'customer_state']


In [8]:
df_model_extended["freight_ratio"] = (
    df_model_extended["total_freight"] / df_model_extended["total_price"]
)

df_model_extended["purchase_month_num"] = pd.to_datetime(
    df_model_extended["purchase_month"], format="%Y-%m"
).dt.month

print(
    df_model_extended[
        ["purchase_month", "purchase_month_num", "customer_state", "freight_ratio"]
    ].head(10)
)

print("\n欠損数")
print(
    df_model_extended[["purchase_month_num", "customer_state", "freight_ratio"]]
    .isna()
    .sum()
)

print("\nfreight_ratioの要約")
print(df_model_extended["freight_ratio"].describe())


  purchase_month  purchase_month_num customer_state  freight_ratio
0        2018-03                 3.0             SP       0.079822
1        2018-05                 5.0             SP       0.060903
2        2018-01                 1.0             RJ       0.413077
3        2017-04                 4.0             AL       0.054740
4        2017-05                 5.0             SP       0.409689
5        2018-05                 5.0             MG       0.423398
6        2017-11                11.0             RJ       0.291584
7        2018-03                 3.0             RJ       0.171006
8        2017-05                 5.0             MG       0.140428
9        2018-08                 8.0             MS       0.453133

欠損数
purchase_month_num    6
customer_state        6
freight_ratio         0
dtype: int64

freight_ratioの要約
count    95830.000000
mean         0.308446
std          0.311781
min          0.000000
25%          0.132028
50%          0.224374
75%          0.380567
m

In [9]:
print(
    df_model_extended[["order_id", "total_price", "total_freight", "freight_ratio"]]
    .sort_values("freight_ratio", ascending=False)
    .head(20)
)


                               order_id  total_price  total_freight  \
63468  6e864b3f0ec71031117ad4cf46b7f2a1         0.85          18.23   
24964  3ee6513ae7ea23bdfab5b9ab60bffcb5         0.85          18.23   
16086  7c92284adbec8033d160303d83065dbf         9.18          54.69   
78531  b2d30dfaf7114de8ab3f737d062032fe         5.95          34.15   
1237   21fb84496c034392ca2aafe7f7919a3f         6.99          37.90   
74530  a16607f4220f0e60ab4c71b915dd380c        14.00          74.08   
71913  95d6357ffe41aa6d2998852a710c70a0        17.50          91.15   
4719   8272b63d03f5f79c56e9e4120aec44ef        31.80         164.37   
38604  f9ccaff7267fd0cf076e795b1fae8b69         3.00          15.23   
2093   3aa41f0c241b0ef62147d849b02313be         4.90          24.84   
65197  765e82006189100419015233bf26be8a         9.99          50.00   
68294  84d54e2fe6fc76e34d897b32c59dd75d         3.49          16.79   
39346  0136390286be8a34efdc801ed150568f         6.98          33.58   
53976 

In [10]:
print(df_model_extended["freight_ratio"].quantile([0.90, 0.95, 0.99, 0.995, 0.999]))


0.900    0.622400
0.950    0.839071
0.990    1.455407
0.995    1.806878
0.999    2.805580
Name: freight_ratio, dtype: float64
